# DQN Reward Shaping

Tests different reward configurations for the DQN agent against the beginner preset (9×9, 10 mines).  
Training uses `data/beginner_train.npz`, evaluation uses the held-out `data/beginner_test.npz`.  
Flag actions are included — action space is doubled: `0..80` = reveal, `81..161` = flag/unflag.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "imageio"], check=True)

sys.path.insert(0, "..")

import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
from dataclasses import dataclass
from IPython.display import Image as IPImage, display

from src.env.minesweeper_env import MinesweeperEnv
from src.train.dqn import DQNAgent

print(f"torch {torch.__version__}")
print("All imports OK")

## Reward Config

In [ ]:
@dataclass
class RewardConfig:
    # Safe cell reveal
    reveal_reward:       float = 1.0     # per newly revealed safe cell

    # Mine hit
    mine_penalty:        float = -10.0   # base penalty
    mine_variant:        str   = "fixed" # "fixed" | "info_scaled" | "progressive"
    #   fixed:        mine_penalty always
    #   info_scaled:  mine_penalty * (1 + revealed_neighbours/8)  — harsher when you had clues
    #   progressive:  mine_penalty * (step/max_steps)             — harsher later in game

    # Win
    win_bonus:           float = 50.0
    win_variant:         str   = "fixed" # "fixed" | "efficiency"
    #   fixed:       win_bonus always
    #   efficiency:  win_bonus * (1 - steps/max_steps)  — bigger reward for winning faster

    # Per-step survival bonus
    step_reward:         float = 0.0     # 0 = off, e.g. +0.1 per step survived

    # Flag rewards (immediate, on each flag action)
    correct_flag_reward: float = 0.0     # reward for flagging a mine
    wrong_flag_penalty:  float = 0.0     # penalty for flagging a safe cell

    max_steps:           int   = 200     # used by progressive/efficiency variants

## Reward Wrapper

Wraps `MinesweeperEnv` to:
- Recompute rewards per `RewardConfig`
- Add flag actions (`N..2N-1`) alongside reveal actions (`0..N-1`)

In [ ]:
class RewardWrapper:
    def __init__(self, env: MinesweeperEnv, cfg: RewardConfig):
        self._env  = env
        self.cfg   = cfg
        self.rows  = env.rows
        self.cols  = env.cols
        self.action_size = env.action_size * 2  # reveal + flag
        self.state_size  = env.state_size
        self._step = 0

    def reset(self):
        self._step = 0
        return self._env.reset()

    def get_valid_actions(self):
        board   = self._env._game.board
        N       = self._env.action_size
        hidden  = board.hidden_cells()  # valid reveals
        flagged = [i for i, v in enumerate(board.view.flatten()) if v == 9]
        # can flag any hidden cell, or unflag any flagged cell
        flag_actions = [i + N for i in hidden + flagged]
        return hidden + flag_actions

    def step(self, action):
        N     = self._env.action_size
        board = self._env._game.board
        self._step += 1

        if action >= N:
            # ---- Flag action ----
            cell     = action - N
            row, col = divmod(cell, self.cols)
            was_flagged = (board.view[row, col] == 9)
            self._env._game.flag(row, col)
            now_flagged = (board.view[row, col] == 9)

            reward = 0.0
            if board._mines_placed and now_flagged and not was_flagged:
                # just placed a new flag — give immediate feedback
                if board.is_mine(row, col):
                    reward = self.cfg.correct_flag_reward
                else:
                    reward = -self.cfg.wrong_flag_penalty

            obs  = self._env._game.board.get_observation()
            done = self._env._game.is_over
            info = {
                "state":      self._env._game.state.name,
                "result":     "flag",
                "mines_left": board.mines_remaining(),
            }
            return obs, reward, done, info

        # ---- Reveal action ----
        obs_before = self._env._game.board.get_observation()
        obs, _, done, info = self._env.step(action)

        newly_revealed = (
            int(((obs >= 0) & (obs <= 8)).sum()) -
            int(((obs_before >= 0) & (obs_before <= 8)).sum())
        )

        reward = 0.0
        if info["result"] == "mine":
            r = self.cfg.mine_penalty
            if self.cfg.mine_variant == "info_scaled":
                row, col = divmod(action, self.cols)
                adj = sum(
                    1 for dr in [-1, 0, 1] for dc in [-1, 0, 1]
                    if (dr or dc)
                    and 0 <= row + dr < self.rows
                    and 0 <= col + dc < self.cols
                    and board.view[row + dr, col + dc] >= 0
                )
                r *= (1 + adj / 8)
            elif self.cfg.mine_variant == "progressive":
                r *= (self._step / self.cfg.max_steps)
            reward += r

        elif info["result"] != "already_revealed":
            reward += newly_revealed * self.cfg.reveal_reward
            reward += self.cfg.step_reward
            if done and info["state"] == "WON":
                wb = self.cfg.win_bonus
                if self.cfg.win_variant == "efficiency":
                    wb *= max(0.0, 1 - self._step / self.cfg.max_steps)
                reward += wb

        return obs, reward, done, info

## Training Function

In [ ]:
def run_experiment(
    cfg,
    n_episodes     = 20_000,
    train_dataset  = "../data/beginner_train.npz",
    eval_dataset   = "../data/beginner_test.npz",
    eval_every     = 500,
    eval_episodes  = 200,
    verbose        = True,
):
    base_env = MinesweeperEnv(preset="beginner", dataset=train_dataset)
    env      = RewardWrapper(base_env, cfg)
    agent    = DQNAgent(
        state_size=env.state_size,
        action_size=env.action_size,
    )

    train_wins, train_rewards, train_lengths = [], [], []
    eval_win_rates  = []   # test-set win rate, logged every eval_every episodes
    loss_log        = []   # mean loss per episode

    for ep in range(n_episodes):
        obs  = env.reset()
        done = False
        ep_reward, ep_steps, ep_losses = 0.0, 0, []

        while not done:
            action = agent.choose_action(obs, env.get_valid_actions())
            next_obs, reward, done, info = env.step(action)
            agent.push(obs, action, reward, next_obs, done)
            loss = agent.train_step()
            if loss is not None:
                ep_losses.append(loss)
            obs        = next_obs
            ep_reward += reward
            ep_steps  += 1

        train_wins.append(info["state"] == "WON")
        train_rewards.append(ep_reward)
        train_lengths.append(ep_steps)
        loss_log.append(float(np.mean(ep_losses)) if ep_losses else float("nan"))

        # ---- Test-set evaluation ----
        if (ep + 1) % eval_every == 0:
            saved_eps = agent.epsilon
            agent.epsilon = 0.0  # greedy eval
            eval_base = MinesweeperEnv(preset="beginner", dataset=eval_dataset)
            eval_env  = RewardWrapper(eval_base, cfg)
            wins = 0
            for _ in range(eval_episodes):
                o, d = eval_env.reset(), False
                while not d:
                    a = agent.choose_action(o, eval_env.get_valid_actions())
                    o, _, d, i = eval_env.step(a)
                wins += i["state"] == "WON"
            agent.epsilon = saved_eps
            eval_win_rates.append(wins / eval_episodes)
            if verbose:
                train_wr = np.mean(train_wins[-eval_every:])
                print(f"  ep {ep+1:>6}  train_wr={train_wr:.3f}  "
                      f"test_wr={eval_win_rates[-1]:.3f}  ε={agent.epsilon:.3f}")

    return {
        "train_wins":     train_wins,
        "train_rewards":  train_rewards,
        "train_lengths":  train_lengths,
        "eval_win_rates": eval_win_rates,
        "loss_log":       loss_log,
        "agent":          agent,
    }

## Ablation Configs

Each config isolates one reward component. Run the cell below to train all configs sequentially (takes a while — reduce `n_episodes` for a quick test).

In [ ]:
CONFIGS = {
    "A_baseline":         RewardConfig(reveal_reward=0.0, step_reward=0.0),
    "B_flat_reveal":      RewardConfig(reveal_reward=1.0),
    "C_info_mine":        RewardConfig(reveal_reward=1.0, mine_variant="info_scaled"),
    "D_progressive_mine": RewardConfig(reveal_reward=1.0, mine_variant="progressive"),
    "E_efficiency_win":   RewardConfig(reveal_reward=1.0, win_variant="efficiency"),
    "F_survival":         RewardConfig(reveal_reward=1.0, step_reward=0.1),
    "G_flags":            RewardConfig(reveal_reward=1.0, correct_flag_reward=3.0,
                                       wrong_flag_penalty=3.0),
    "H_all":              RewardConfig(reveal_reward=1.0, mine_variant="info_scaled",
                                       win_variant="efficiency", step_reward=0.05,
                                       correct_flag_reward=3.0, wrong_flag_penalty=3.0),
}

N_EPISODES = 20_000   # reduce to e.g. 5_000 for a quick smoke test

results = {}
for name, cfg in CONFIGS.items():
    print(f"\n{'='*56}")
    print(f"  Config: {name}")
    print(f"{'='*56}")
    results[name] = run_experiment(cfg, n_episodes=N_EPISODES)

print("\nAll configs done.")

## Comparison Plots

In [ ]:
os.makedirs("../recordings", exist_ok=True)

WINDOW    = 200
EVAL_EVERY = 500
COLOURS = plt.cm.tab10(np.linspace(0, 1, len(CONFIGS)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("DQN Reward Shaping Ablation — Beginner Minesweeper", fontsize=14)

# 1 — Rolling training win rate
ax = axes[0, 0]
for (name, res), col in zip(results.items(), COLOURS):
    wins = np.array(res["train_wins"], dtype=float)
    if len(wins) >= WINDOW:
        rolling = np.convolve(wins, np.ones(WINDOW) / WINDOW, mode="valid")
        ax.plot(range(WINDOW - 1, len(wins)), rolling * 100, label=name, color=col, lw=1.2)
ax.set_title(f"Training Win Rate (rolling {WINDOW}-ep)")
ax.set_xlabel("Episode")
ax.set_ylabel("Win rate (%)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# 2 — Test-set eval win rate over training
ax = axes[0, 1]
for (name, res), col in zip(results.items(), COLOURS):
    evals = res["eval_win_rates"]
    xs    = [(i + 1) * EVAL_EVERY for i in range(len(evals))]
    ax.plot(xs, [e * 100 for e in evals], label=name, color=col, lw=1.5, marker="o", ms=3)
ax.set_title("Test Set Win Rate (greedy eval every 500 eps)")
ax.set_xlabel("Training episode")
ax.set_ylabel("Test win rate (%)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# 3 — Final test win rate bar chart
ax = axes[1, 0]
final_test_wrs = {name: res["eval_win_rates"][-1] * 100
                  for name, res in results.items() if res["eval_win_rates"]}
bars = ax.bar(final_test_wrs.keys(), final_test_wrs.values(),
              color=COLOURS[:len(final_test_wrs)], edgecolor="white")
for bar, val in zip(bars, final_test_wrs.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_title("Final Test Win Rate (last eval checkpoint)")
ax.set_ylabel("Win rate (%)")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, alpha=0.3, axis="y")

# 4 — Avg episode length (last 1000 training eps)
ax = axes[1, 1]
tail = 1000
avg_lengths = {name: np.mean(res["train_lengths"][-tail:])
               for name, res in results.items()}
bars = ax.bar(avg_lengths.keys(), avg_lengths.values(),
              color=COLOURS[:len(avg_lengths)], edgecolor="white")
for bar, val in zip(bars, avg_lengths.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8)
ax.set_title(f"Avg Episode Length (last {tail} training eps)")
ax.set_ylabel("Steps")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../recordings/dqn_ablation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/dqn_ablation.png")

## Best Config: Detailed Analysis

In [ ]:
# Pick best config by final test win rate
best_name = max(results, key=lambda n: results[n]["eval_win_rates"][-1]
                if results[n]["eval_win_rates"] else 0)
best = results[best_name]
print(f"Best config: {best_name}  "
      f"(final test win rate: {best['eval_win_rates'][-1]*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"Best Config: {best_name}", fontsize=13)

# Loss curve
ax = axes[0]
losses = [l for l in best["loss_log"] if not np.isnan(l)]
w = min(200, len(losses) // 10 or 1)
smooth = np.convolve(losses, np.ones(w) / w, mode="valid")
ax.plot(smooth, color="steelblue", lw=1)
ax.set_title("Training Loss (smoothed)")
ax.set_xlabel("Episode")
ax.set_ylabel("MSE Loss")
ax.grid(True, alpha=0.3)

# Episode reward distribution (last 5000 eps)
ax = axes[1]
tail_rewards = best["train_rewards"][-5000:]
ax.hist(tail_rewards, bins=40, color="mediumpurple", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(tail_rewards), color="orange", lw=1.5, ls="--",
           label=f"Mean: {np.mean(tail_rewards):.1f}")
ax.set_title("Episode Reward (last 5k eps)")
ax.set_xlabel("Total reward")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

# Train vs test win rate comparison
ax = axes[2]
wins_arr = np.array(best["train_wins"], dtype=float)
roll_w = 500
if len(wins_arr) >= roll_w:
    rolling_train = np.convolve(wins_arr, np.ones(roll_w) / roll_w, mode="valid")
    ax.plot(range(roll_w - 1, len(wins_arr)), rolling_train * 100,
            label="Train (rolling 500)", color="steelblue", lw=1.2)
eval_xs = [(i + 1) * EVAL_EVERY for i in range(len(best["eval_win_rates"]))]
ax.plot(eval_xs, [e * 100 for e in best["eval_win_rates"]],
        label="Test (greedy)", color="tomato", lw=1.5, marker="o", ms=4)
ax.set_title("Train vs Test Win Rate")
ax.set_xlabel("Episode")
ax.set_ylabel("Win rate (%)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/dqn_best_config.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/dqn_best_config.png")

## GIF Recording — Best Config

In [ ]:
import pygame
pygame.init()

from src.game.renderer import Renderer

GIF_PATH   = "../recordings/dqn_episode.gif"
best_agent = best["agent"]
best_cfg   = CONFIGS[best_name]

# Run on a test board, greedy policy
saved_eps       = best_agent.epsilon
best_agent.epsilon = 0.0

rec_base = MinesweeperEnv(preset="beginner", dataset="../data/beginner_test.npz")
rec_env  = RewardWrapper(rec_base, best_cfg)
obs      = rec_env.reset()

rec_renderer = Renderer(rec_base._game, preset="beginner")

frames = []

def capture():
    rec_renderer._draw()
    return pygame.surfarray.array3d(rec_renderer.screen).transpose(1, 0, 2).copy()

frames.append(capture())

done = False
rec_steps = 0
while not done:
    action = best_agent.choose_action(obs, rec_env.get_valid_actions())
    obs, reward, done, info = rec_env.step(action)
    rec_steps += 1
    frames.append(capture())

best_agent.epsilon = saved_eps

print(f"Recorded: {rec_steps} steps, outcome={info['state']}")
print(f"Frames: {len(frames)}")

imageio.mimsave(GIF_PATH, frames, duration=0.4, loop=0)
print(f"Saved: {GIF_PATH}")

display(IPImage(filename=GIF_PATH))

## Interpretation

### Reward Component Reference

| Config | Components active |
|--------|-------------------|
| A — baseline | mine penalty + win bonus only |
| B — flat reveal | + flat +1 per revealed cell |
| C — info mine | + mine penalty scales with revealed neighbours |
| D — progressive mine | + mine penalty scales with time step |
| E — efficiency win | + win bonus decays with steps taken |
| F — survival | + +0.1 per step survived |
| G — flags | + immediate reward/penalty for correct/wrong flags |
| H — all | all of the above combined |

### What to look for

- **Train vs test gap**: if train win rate climbs but test plateaus, the agent is overfitting to training boards — consider more episodes or a larger dataset.
- **Config A ≈ random baseline (~3–5%)**: confirms the DQN isn't learning from mine penalty alone — reveal reward is necessary.
- **Loss curve**: should generally decrease; if it spikes or diverges, reduce `lr` or increase `buffer_size`.
- **Episode length**: longer episodes = agent surviving longer = good coverage before dying. Short flat lengths = dying fast every game.
- **Flag configs (G, H)**: flag rewards only help if the agent actually uses flag actions. If flag precision is low, the penalty may need to be higher to discourage random flagging.